In [1]:
import pandas as pd
import numpy as np
import openpyxl

In [2]:
df = pd.read_excel('../data/raw/Base_Case_Modelagem2.xlsx', sheet_name='Base sem performance')

In [3]:
df.head(1)

,ID,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19
0,9,190.110001,M,365.0,0.0,0.0,0.0,0.0,0.02994,0.00478,92.0,349.0,Inativo,Inativo,Inativo,1.0,1.0,Inativo,0.0,55


In [4]:
df.drop(columns=['ID'], inplace=True)

In [5]:
df.shape

(16827, 19)

In [6]:
from sklearn.impute import SimpleImputer

num_cols = df.select_dtypes(include=np.number).columns
print(num_cols)

Index(['V1', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V15',
       'V16', 'V18', 'V19'],
      dtype='object')


In [7]:
imputer_num = SimpleImputer(strategy='median')

In [8]:
df[num_cols] = imputer_num.fit_transform(df[num_cols])

In [9]:
cat_cols = df.select_dtypes(include="object").columns
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

In [10]:
df.isnull().sum()

V1     0
V2     0
V3     0
V4     0
V5     0
V6     0
V7     0
V8     0
V9     0
V10    0
V11    0
V12    0
V13    0
V14    0
V15    0
V16    0
V17    0
V18    0
V19    0
dtype: int64

In [12]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

resultado = encoder.fit_transform(df[cat_cols])


In [13]:
df_cat = pd.DataFrame(
    resultado,
    columns=encoder.get_feature_names_out(cat_cols),
    index=df.index
)

In [14]:
df = pd.concat([df.drop(columns=cat_cols), df_cat], axis=1)

In [15]:
df.shape


(16827, 41)

In [18]:
selected_features = [
    "V1",
    "V10",
    "V11",
    "V17_Premium",
    "V18",
    "V19",
    "V2_B",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9"
]

df = df[selected_features]

In [19]:
import joblib

model = joblib.load(
    "../models/lightgbm_final.pkl"
)

selected_features = joblib.load(
    "../models/selected_features.pkl"
)

In [20]:
probabilidade = model.predict_proba(df)[:, 1]

previsao = (probabilidade >= 0.5).astype(int)

In [21]:
df["PROBABILIDADE_INADIMPLENCIA"] = probabilidade
df["PREVISAO"] = previsao

In [22]:
df.head()

,V1,V10,V11,V17_Premium,V18,V19,V2_B,V5,V6,V7,V8,V9,PROBABILIDADE_INADIMPLENCIA,PREVISAO
0,190.110001,92.0,349.0,0.0,0.00000,55.0,0.0,0.000000,0.00000,0.000000,0.02994,0.00478,0.408092,0
1,80.919998,92.0,118.0,0.0,0.00000,59.0,0.0,0.000000,0.00000,0.000000,0.02170,0.07194,0.589739,1
2,124.430000,14.0,344.0,1.0,0.59314,30.0,1.0,394.220001,1.87919,233.828186,0.02170,0.17873,0.487746,0
3,37.785450,2.0,146.0,0.0,0.76424,70.0,1.0,106.489998,2.15385,81.384064,0.02170,0.18832,0.124319,0
4,702.005737,27.0,327.0,1.0,0.52055,43.0,0.0,1727.599976,1.28105,899.301453,0.02170,0.13450,0.121308,0


In [23]:
df.to_excel('../data/results/Base_Sem_Performance_Previsto.xlsx', index=False)